In [ ]:
from pathlib import Path
import win32com.client
import tarfile
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
def get_gdrive_experiment_root(exp_id: str) -> Path:
    tbird_shortcut_path = Path("G:/") / "My Drive" / "Thunderbird Files.lnk"
    shell = win32com.client.Dispatch("WScript.Shell")
    shortcut = shell.CreateShortcut(str(tbird_shortcut_path))
    tbird_files_path = Path(shortcut.Targetpath)
    return tbird_files_path / "0.Experiments" / exp_id

In [ ]:
exp_id = "ID-423"
exp_root = get_gdrive_experiment_root(exp_id)

In [ ]:
exp_tarfiles = [
    x for x in exp_root.iterdir()
    if '.tar' in x.suffixes and '.gz' in x.suffixes
]
exp_tarfile_path = exp_tarfiles[0]

In [ ]:
dfs = {}
with tarfile.open(exp_tarfile_path, "r:gz") as exp_tarfile:
    # print(exp_tarfile.getmembers())
    # exp_tarfile.list()
    for reactor_data_file_info in exp_tarfile.getmembers():
        try:
            df = pd.read_csv(exp_tarfile.extractfile(reactor_data_file_info))  # TODO fix why this can't extract using utf-8 encoding
        except UnicodeDecodeError:
            df = pd.read_csv(exp_tarfile.extractfile(reactor_data_file_info), encoding="cp1252")
        dfs[reactor_data_file_info.name] = df

In [ ]:
def get_reactor_data_df(data_type: str) -> pd.DataFrame:
    df = [v for k, v in dfs.items() if data_type in k][0]
    df["Timestamp"] = pd.to_datetime(df["Timestamp"])
    return df


def add_elapsed_hours_series(dataframe: pd.DataFrame) -> pd.DataFrame:
    start_time = dataframe.iloc[0]["Timestamp"]
    timedelta = dataframe["Timestamp"] - start_time
    dataframe["Elapsed (hours)"] = timedelta.dt.total_seconds() / (60*60)
    return dataframe

In [ ]:
mw_forward_power_df = get_reactor_data_df("Forward Power")
mw_forward_power_df = add_elapsed_hours_series(mw_forward_power_df)

In [ ]:
target_current_df = get_reactor_data_df("Target Current")
target_current_df = add_elapsed_hours_series(target_current_df)

In [ ]:
def correct_current_units(row):
    value = row["Data"]
    unit = row["Units"]
    if unit == "uA":
        return value * 1e-6
    elif unit == "mA":
        return value * 1e-3
    else:
        return value


target_current_df["Data (A)"] = target_current_df.apply(
    correct_current_units, axis=1
)

In [ ]:
x = mw_forward_power_df['Elapsed (hours)']
y = mw_forward_power_df['Data']

fig, ax = plt.subplots()
ax.plot(x, y)
ax.set_xlabel('Elapsed time (hours)')
ax.set_ylabel('Microwave Generator Forward Power (W)')
fig.show()

In [ ]:
x = target_current_df['Elapsed (hours)']
y = target_current_df['Data (A)'] * 1e3

fig, ax = plt.subplots()
ax.plot(x, y)
ax.set_xlabel('Elapsed time (hours)')
ax.set_ylabel('Target Current (mA)')
fig.show()